In [12]:
import numpy as np

def evaluate_traffic_delay(green_times):
    g1, g2 = green_times
    if g1 < 10 or g1 > 120 or g2 < 10 or g2 > 120:
        return 1e6

    arrival_rate = 0.5
    delay1 = max(0, (1 - g1 * arrival_rate) * 50)
    delay2 = max(0, (1 - g2 * arrival_rate) * 50)
    total_waiting_time = delay1 + delay2
    return total_waiting_time

# PSO Parameters
num_particles = 15
max_iterations = 30
w = 0.5      # Inertia weight
c1 = 1.5     # Cognitive (personal best) weight
c2 = 1.5     # Social (global best) weight

# Search space bounds (min and max green time in seconds for 2 traffic phases)
lb = np.array([10.0, 10.0])
ub = np.array([120.0, 120.0])
dim = 2

# Initialize particle positions and velocities
positions = np.random.uniform(lb, ub, (num_particles, dim))
velocities = np.zeros((num_particles, dim))

# Initialize personal bests
pbest_positions = np.copy(positions)
pbest_scores = np.array([evaluate_traffic_delay(p) for p in positions])

# Initialize global best
gbest_idx = np.argmin(pbest_scores)
gbest_position = np.copy(pbest_positions[gbest_idx])
gbest_score = pbest_scores[gbest_idx]

# Main PSO Loop
for iteration in range(max_iterations):
    for i in range(num_particles):
        r1, r2 = np.random.rand(2)

        # Update velocity
        velocities[i] = (w * velocities[i] +
                         c1 * r1 * (pbest_positions[i] - positions[i]) +
                         c2 * r2 * (gbest_position - positions[i]))

        # Update position
        positions[i] += velocities[i]

        # Clamp positions to search space bounds
        positions[i] = np.clip(positions[i], lb, ub)

        # Evaluate new position
        score = evaluate_traffic_delay(positions[i])

        # Update personal best
        if score < pbest_scores[i]:
            pbest_scores[i] = score
            pbest_positions[i] = np.copy(positions[i])

        # Update global best
        if score < gbest_score:
            gbest_score = score
            gbest_position = np.copy(positions[i])

print(f"Optimal Green Light Times (Phase 1, Phase 2): {gbest_position}")
print(f"Minimum Estimated Total Delay: {gbest_score:.2f} seconds")


Optimal Green Light Times (Phase 1, Phase 2): [57.65440867 59.08289394]
Minimum Estimated Total Delay: 0.00 seconds


In [9]:
import numpy as np

def evaluate_traffic_delay(green_times):
    g1, g2 = green_times
    # Strictly enforce search boundaries
    if g1 < 10 or g1 > 120 or g2 < 10 or g2 > 120:
        return 1e6

    # Yellow/Clearance time for safety between transitions
    yellow_time = 4.0

    # Total cycle time = Green 1 + Yellow + Green 2 + Yellow
    cycle_length = g1 + g2 + (2 * yellow_time)

    # Red light times (One phase is red while the other phase + yellows are active)
    r1 = g2 + (2 * yellow_time)
    r2 = g1 + (2 * yellow_time)

    # Traffic arrival rates (cars arriving per second)
    arrival_rate1 = 0.25
    arrival_rate2 = 0.20

    # Calculate vehicular queue accumulation during the Red Light phase
    cars_waiting_phase1 = r1 * arrival_rate1
    cars_waiting_phase2 = r2 * arrival_rate2


    delay1 = 0.5 * cars_waiting_phase1 * r1
    delay2 = 0.5 * cars_waiting_phase2 * r2

    return delay1 + delay2

# PSO Parameters
num_particles = 15
max_iterations = 30
w = 0.5   # Inertia weight
c1 = 1.5  # Cognitive (personal best) weight
c2 = 1.5  # Social (global best) weight

# Search space bounds (Min and Max green times allowed)
lb = np.array([10.0, 10.0])
ub = np.array([120.0, 120.0])
dim = 2

# Initialize particle positions and velocities
positions = np.random.uniform(lb, ub, (num_particles, dim))
velocities = np.zeros((num_particles, dim))

# Initialize personal bests
pbest_positions = np.copy(positions)
pbest_scores = np.array([evaluate_traffic_delay(p) for p in positions])

# Initialize global best
gbest_idx = np.argmin(pbest_scores)
gbest_position = np.copy(pbest_positions[gbest_idx])
gbest_score = pbest_scores[gbest_idx]

# Main PSO Loop
for iteration in range(max_iterations):
    for i in range(num_particles):
        r1_rand, r2_rand = np.random.rand(2)

        # Update velocity
        velocities[i] = (w * velocities[i] +
                         c1 * r1_rand * (pbest_positions[i] - positions[i]) +
                         c2 * r2_rand * (gbest_position - positions[i]))

        # Update position
        positions[i] += velocities[i]

        # Clamp positions to search space bounds
        positions[i] = np.clip(positions[i], lb, ub)

        # Evaluate new position
        score = evaluate_traffic_delay(positions[i])

        # Update personal best
        if score < pbest_scores[i]:
            pbest_scores[i] = score
            pbest_positions[i] = np.copy(positions[i])

        # Update global best
        if score < gbest_score:
            gbest_score = score
            gbest_position = np.copy(positions[i])


print(f"Optimal Green Light Times (Phase 1, Phase 2): [{gbest_position[0]:.2f}s, {gbest_position[1]:.2f}s]")
print(f"Minimum Estimated Total Delay: {gbest_score:.2f} car-seconds per cycle")
